# ReconMind Stage 3, Model B: predict the ReconMind test set

This notebook takes the fine tuned SciBERT model from training run 2 (`train_attack_mapper.ipynb`) and asks it which MITRE ATT&CK techniques each ReconMind finding describes. The findings are real scan results from Metasploitable 2 and pfSenseLabB, stored in `evaluation/data/reconmind_eval/findings.jsonl`.

Nothing is trained here. The notebook only loads the saved model from your Google Drive and predicts. It uses the same threshold and maximum sentence length that run 2 chose on its validation split, so the test set never influences any setting.

The answer key (`labels.jsonl`) is not read here. Scoring happens on your own computer with `evaluation/scripts/score_reconmind_exam.py`, which scores Model A and Model B with the same code.

**How to run:** a CPU is fine, there are only a few dozen findings. Choose Runtime, Run all. Google asks for permission to use your Drive, where the run 2 model was saved. At the end your browser downloads `finetuned_scibert_run2_reconmind_exam.jsonl`. Put it in `evaluation/results/raw/` of your local ReconMind copy.

The notebook clones the `attack-mapper` branch from GitHub, so that branch must already contain `evaluation/data/reconmind_eval/findings.jsonl`.

## 1. Get the ReconMind code

Downloads the project from GitHub (branch `attack-mapper`), the same way the training notebook does, so we get the findings file and the helper that turns probabilities into technique IDs. The commit ID is printed so the predictions can be traced to the exact code and findings that produced them.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/r00tSud0/ReconMind'
BRANCH = 'attack-mapper'
REPO_DIR = '/content/ReconMind'

if os.path.isdir(REPO_DIR):
    print('Already cloned, reusing', REPO_DIR)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR],
                   check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

CODE_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], check=True,
                             capture_output=True, text=True).stdout.strip()
print('code commit:', CODE_COMMIT)

## 2. Install the Hugging Face library

`transformers` loads the saved SciBERT model and its tokenizer. PyTorch already comes with Colab. The notebook uses a GPU if one is attached and the CPU otherwise; the results are the same either way, up to tiny rounding differences.

In [ ]:
!pip install -q transformers

import torch
import transformers

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| transformers', transformers.__version__, '| device', DEVICE)

## 3. Load the run 2 model from Google Drive

Connects your Drive (a popup asks for permission) and loads the model and tokenizer that training run 2 saved to `MyDrive/ReconMind/scibert_attack_mapper_run2`.

The model gives 50 scores, one per technique, in a fixed order. That order comes from the `labels.json` saved in the same folder. As a safety check, the notebook confirms it matches the order stored inside the model itself, so a score can never be read under the wrong technique ID.

In [ ]:
import json

from google.colab import drive
from transformers import AutoModelForSequenceClassification, AutoTokenizer

drive.mount('/content/drive')
MODEL_DIR = '/content/drive/MyDrive/ReconMind/scibert_attack_mapper_run2'
if not os.path.isdir(MODEL_DIR):
    raise RuntimeError('Cannot find ' + MODEL_DIR + '. Run train_attack_mapper.ipynb '
                       '(run 2) first; it saves the model there.')

with open(os.path.join(MODEL_DIR, 'labels.json')) as fh:
    LABELS = [t['id'] if isinstance(t, dict) else t for t in json.load(fh)]
assert len(LABELS) == 50, 'expected 50 labels, found {n}'.format(n=len(LABELS))

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(DEVICE)
model.eval()

model_order = [model.config.id2label[i] for i in range(model.config.num_labels)]
if model_order != LABELS:
    raise RuntimeError('The label order in labels.json does not match the model. Stopping.')
print('loaded model from', MODEL_DIR, 'with', len(LABELS), 'labels')

## 4. Read the threshold and maximum length chosen in training

A technique is predicted when its probability is at least the threshold. Run 2 picked one threshold for all 50 techniques on its validation split, and also picked the maximum number of tokens per sentence. Both are read from `finetuned_scibert_run2_metrics.json` in the Drive folder, so this notebook uses exactly the settings that were scored on the TRAM eval set.

In [ ]:
with open(os.path.join(MODEL_DIR, 'finetuned_scibert_run2_metrics.json')) as fh:
    TRAINING = json.load(fh)

THRESHOLD = TRAINING['threshold']
MAX_LENGTH = TRAINING['max_length']
print('threshold:', THRESHOLD, '| max length:', MAX_LENGTH,
      '| trained from code commit:', TRAINING.get('code_commit'))

## 5. Read the findings

Each line of `findings.jsonl` is one open port from a real scan. Only its `text` field (the plain English description ReconMind writes) goes to the model, the same input Model A gets.

In [ ]:
FINDINGS_PATH = os.path.join(REPO_DIR, 'evaluation', 'data', 'reconmind_eval', 'findings.jsonl')
with open(FINDINGS_PATH) as fh:
    findings = [json.loads(line) for line in fh if line.strip()]

print(len(findings), 'findings')
print('first one:', findings[0]['text'])

## 6. Predict

Runs the model without learning, a few findings at a time, and turns each of the 50 scores into a probability between 0 and 1 (the sigmoid function). Every technique whose probability is at or above the threshold becomes a prediction, using `predict_ids` from `evaluation/attack_mapper/finetune_utils.py`, the same rule used on the TRAM eval set.

In [ ]:
from evaluation.attack_mapper import finetune_utils

BATCH_SIZE = 16


@torch.no_grad()
def predict_probs(texts):
    probs = []
    for start in range(0, len(texts), BATCH_SIZE):
        batch = tokenizer(texts[start:start + BATCH_SIZE], truncation=True,
                          max_length=MAX_LENGTH, padding=True, return_tensors='pt')
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        logits = model(**batch).logits
        probs.extend(torch.sigmoid(logits.float()).cpu().tolist())
    return probs


all_probs = predict_probs([f['text'] for f in findings])
all_preds = [finetune_utils.predict_ids(p, LABELS, THRESHOLD) for p in all_probs]

for f, pred in zip(findings, all_preds):
    print('{i:<22} {p}'.format(i=f['id'], p=', '.join(pred) if pred else '(none)'))

## 7. Save the predictions

Writes one line per finding to `evaluation/results/raw/finetuned_scibert_run2_reconmind_exam.jsonl` with the finding `id`, the predicted IDs (`pred`) and all 50 probabilities (`probs`, rounded to 4 places, in the `labels.json` order). Keeping the probabilities means a different threshold can be checked later without running the model again.

In [ ]:
RAW_DIR = os.path.join(REPO_DIR, 'evaluation', 'results', 'raw')
os.makedirs(RAW_DIR, exist_ok=True)
OUT_PATH = os.path.join(RAW_DIR, 'finetuned_scibert_run2_reconmind_exam.jsonl')

with open(OUT_PATH, 'w') as fh:
    for f, pred, probs in zip(findings, all_preds, all_probs):
        fh.write(json.dumps({'id': f['id'], 'pred': pred,
                             'probs': [round(p, 4) for p in probs]}) + '\n')

print('wrote', len(findings), 'lines to', OUT_PATH)

## 8. Download the predictions

Offers the file as a browser download. Put it in the `evaluation/results/raw/` folder of your local ReconMind copy, then run from the repo root:

```
python3 evaluation/scripts/score_reconmind_exam.py
```

In [ ]:
from google.colab import files

files.download(OUT_PATH)